# 6.4 基于langgraph的多无人机协同搜索

# 代码演练：构建多智能体系统

现在，我们将通过一个详细的代码演练来构建我们的多智能体系统。

## 1. 定义图的状态 (Graph State)

图的状态是整个系统的“单一事实来源”（Single Source of Truth），它是一个在所有节点之间传递和更新的共享内存对象。

我们使用 Python 的 `TypedDict` 来定义它的结构，以确保状态的字段类型清晰、可维护，并能被 LangGraph 正确识别和追踪。

In [6]:
from typing import TypedDict, Annotated, List, Dict
from langchain_core.messages import BaseMessage
import operator

class GraphState(TypedDict):
    """
    ... (docstring不变)
    """
    mission_goal: str
    
    # 关键修复：task_queue 无 Annotated（替换整个队列）
    task_queue: List[Dict]  
    # completed_tasks 保留 Annotated（追加新任务）
    completed_tasks: Annotated[List[Dict], operator.add]
    
    # messages 保留
    messages: Annotated[List[BaseMessage], operator.add]

## 2. 构建监督者节点

监督者节点是一个Python函数，它接收当前的GraphState作为输入，其核心逻辑由一个LLM驱动。

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

# 使用火山云模型（兼容 OpenAI 接口）
llm = ChatOpenAI(
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key="80e68c38-22cb-4f71-9377-0768c4d7fe15",
    model="doubao-seed-1-6-250615",  # 替换为你的模型 ID
    temperature=0.01,
)

def supervisor_node(state: GraphState):
    """
    监督者节点，负责任务分解和分配。
    """
    print("---进入监督者节点---")
    
    # 如果任务队列为空，说明是第一次运行，需要分解任务
    if not state['task_queue']:
        system_prompt = (
            "你是一个无人机集群的任务指挥官。你的任务是接收一个高层目标，"
            "并将其分解为一系列分配给两架无人机（Drone1, Drone2）的具体航点任务。"
            "每个任务应该是一个包含'drone_id'和'target'（一个包含x, y, z坐标的列表）的字典。"
            "将所有任务以JSON列表的形式输出，不要包含其他任何文字。"
            "Z坐标应为负数表示高度，例如-10表示10米高。"
        )
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "任务目标: {mission_goal}")
        ])
        
        chain = prompt | llm
        response = chain.invoke({"mission_goal": state['mission_goal']})
        
        import json
        try:
            tasks = json.loads(response.content)
            # 确保是列表
            if isinstance(tasks, list):
                print(f"监督者分解出以下任务: {tasks}")
            else:
                tasks = []  # fallback
                print("警告: LLM 输出非列表，任务为空")
        except json.JSONDecodeError:
            tasks = []  # fallback
            print("警告: LLM 输出非有效 JSON，任务为空")
        
        # 更新状态：返回新任务列表（Annotated 会追加，但首次为空所以设置）
        return {"task_queue": tasks, "messages": [HumanMessage(content=f"任务已分解: {tasks}")]}
    else:
        # 如果任务队列不为空，则无需操作
        return {}

我们为监督者LLM精心设计了一个系统提示，明确了它的角色、输入和期望的输出格式（JSON），这极大地提高了其输出的可靠性。

## 3. 构建侦察员节点

侦察员节点代表了无人机的实际行动。它从任务队列中取出一个任务并执行。

In [8]:
from langchain_core.messages import AIMessage, ToolMessage  # 新增 ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from airsim_agent import *  # takeoff, fly_to_position

# tool_node 不变
tool_node = ToolNode([takeoff, fly_to_position])

def scout_node(state: GraphState):
    """
    侦察员节点，负责执行单个任务。
    """
    print("---进入侦察员节点---")
    
    if state['task_queue']:
        task = state['task_queue'][0]  # 取第一个（dict）
        drone_id = task['drone_id']
        target = task['target']  # [x, y, z]
        
        # Prompt 保持强化（.format 注入参数）
        system_prompt = (
            "你是一架自主无人机。你 MUST 严格按照以下步骤调用工具完成任务："
            "1. 先调用 takeoff( drone_id='{drone_id}' ) 起飞。"
            "2. 然后调用 fly_to_position( drone_id='{drone_id}', x={x}, y={y}, z={z} ) 飞往位置。"
            "不要输出任何其他文本，只输出工具调用。参数必须精确匹配。"
        ).format(drone_id=drone_id, x=target[0], y=target[1], z=target[2])
        
        human_input = f"立即执行：{drone_id} 起飞并飞往 {target}。"
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", human_input)
        ])
        
        llm_with_tools = llm.bind_tools([takeoff, fly_to_position])
        chain = prompt | llm_with_tools
        ai_message = chain.invoke({})
        
        # 调试：打印 LLM 输出
        print(f"LLM 输出 for {drone_id}: {ai_message.content[:100]}...")  # 截断查看
        if hasattr(ai_message, 'tool_calls'):
            print(f"Tool calls: {len(ai_message.tool_calls)}")
        
        messages_to_add = [ai_message]
        
        # 新增：如果无工具调用，手动模拟执行（直接调用工具，添加 ToolMessage）
        if not hasattr(ai_message, 'tool_calls') or not ai_message.tool_calls:
            print(f"警告：{drone_id} 未生成工具调用，手动模拟执行。")
            # 手动 takeoff
            takeoff_result = takeoff.invoke({"drone_id": drone_id})
            messages_to_add.append(ToolMessage(content=takeoff_result, tool_call_id="manual_takeoff"))
            # 手动 fly_to
            fly_result = fly_to_position.invoke({"drone_id": drone_id, "x": target[0], "y": target[1], "z": target[2]})
            messages_to_add.append(ToolMessage(content=fly_result, tool_call_id="manual_fly"))
            # 覆盖 ai_message 为模拟（可选，避免 router 误判）
            ai_message = AIMessage(content=f"{drone_id} 手动执行成功。")
            messages_to_add = [ai_message] + messages_to_add[1:]  # 替换
        
        # 状态更新：弹出任务
        remaining_tasks = state['task_queue'][1:]  # 剩余列表
        new_completed = [task]  # 只返回新任务，Annotated 会追加
        
        return {
            "task_queue": remaining_tasks,  # 替换整个队列（无 Annotated）
            "completed_tasks": new_completed,  # 追加单个任务
            "messages": messages_to_add
        }
    else:
        print("所有任务已完成！")
        return {"messages": [HumanMessage(content="所有任务已完成")]}

## 4. 组装图 (Graph)

现在，我们使用LangGraph的StateGraph将所有部分组装起来。

In [9]:
# 创建图
workflow = StateGraph(GraphState)

# 添加节点 (不变)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("scout", scout_node)
workflow.add_node("tool_executor", tool_node)

# 设置入口点 (不变)
workflow.set_entry_point("supervisor")


# 路由：小优化，加调试
def router(state: GraphState):
    print("---进入路由器---")
    
    if not state['messages']:
        print("无消息，结束")
        return END
    
    last_message = state['messages'][-1]
    
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print("检测到工具调用，路由到 tool_executor")
        return "tool_executor"
    
    if state['task_queue']:
        print(f"还有 {len(state['task_queue'])} 个任务，继续 scout")
        return "scout"
    
    print("任务完成，结束")
    return END

# 边：保持 {"tool_executor": "tool_executor", "scout": "scout", END: END}
workflow.add_edge("supervisor", "scout")
workflow.add_conditional_edges(
    "scout", 
    router, 
    {"tool_executor": "tool_executor", "scout": "scout", END: END}
)
workflow.add_edge("tool_executor", "scout")

# 编译
app = workflow.compile()

这里的核心是router函数，它检查图的当前状态来决定流程的下一步走向。这种显式的控制流使得整个系统的行为既强大又可预测。

## 5. 执行与观察

最后，我们提供初始状态并运行我们的多智能体系统。

In [10]:
# 运行：加 max_iterations 防万一
initial_state = {
    "mission_goal": "让Drone1飞到(-10, 10, -10)，让Drone2飞到(10, 10, -10)。",
    "task_queue": [],
    "completed_tasks": [],
    "messages": []
}

for event in app.stream(initial_state, {"recursion_limit": 100, "max_iterations": 20}):  # 新增 max_iterations
    for key, value in event.items():
        print(f"节点: {key}")
        print("---")
        print(value)
    print("\n===================\n")

# 最终检查
final_state = app.invoke(initial_state, config={"max_iterations": 20})
print("最终状态摘要：")
print(f"剩余任务: {len(final_state['task_queue'])}")
print(f"完成任务: {final_state['completed_tasks']}")
print(f"消息数: {len(final_state['messages'])}")

---进入监督者节点---
监督者分解出以下任务: [{'drone_id': 'Drone1', 'target': [-10, 10, -10]}, {'drone_id': 'Drone2', 'target': [10, 10, -10]}]
节点: supervisor
---
{'task_queue': [{'drone_id': 'Drone1', 'target': [-10, 10, -10]}, {'drone_id': 'Drone2', 'target': [10, 10, -10]}], 'messages': [HumanMessage(content="任务已分解: [{'drone_id': 'Drone1', 'target': [-10, 10, -10]}, {'drone_id': 'Drone2', 'target': [10, 10, -10]}]", additional_kwargs={}, response_metadata={})]}


---进入侦察员节点---
LLM 输出 for Drone1: ...
Tool calls: 0
警告：Drone1 未生成工具调用，手动模拟执行。
Executing: takeoff for Drone1
Executing: fly_to_position for Drone1 to (-10.0, 10.0, -10.0)
---进入路由器---
还有 1 个任务，继续 scout
节点: scout
---
{'task_queue': [{'drone_id': 'Drone2', 'target': [10, 10, -10]}], 'completed_tasks': [{'drone_id': 'Drone1', 'target': [-10, 10, -10]}], 'messages': [AIMessage(content='Drone1 手动执行成功。', additional_kwargs={}, response_metadata={}), ToolMessage(content='无人机 Drone1 已经成功起飞。', tool_call_id='manual_takeoff'), ToolMessage(content='无人机 Dron

### 运行结果分析：是的，基本正确！无人机任务已成功执行

哈哈，从日志看，你的系统运行得很顺利——“飞机都飞起来了”就是最好的验证！AirSim 模拟器中看到 Drone1 和 Drone2 起飞并到达目标位置 (Drone1: (-10, 10, -10)，Drone2: (10, 10, -10))，说明工具调用（或手动模拟）生效了。任务队列清空，完成列表有 2 个任务，消息日志记录完整。这证明了 LangGraph 的状态管理和路由逻辑工作正常。下面我一步步拆解日志，确认正确性，并指出小优化点。

#### 1. **整体流程正确性确认**
- **第一次运行（stream）**：
  - **supervisor**：LLM 正确分解使命目标为 2 个任务（Drone1 和 Drone2），`task_queue` 填充成功。返回 HumanMessage 日志。
  - **scout (Drone1)**：LLM 未生成工具调用（Tool calls: 0），触发手动模拟：
    - 调用 `takeoff("Drone1")` → "Executing: takeoff for Drone1"（起飞）。
    - 调用 `fly_to_position("Drone1", -10, 10, -10)` → "Executing: fly_to_position..."（飞行）。
    - 更新状态：`task_queue` 剩 [Drone2]，`completed_tasks` 加 [Drone1]，messages 加 AIMessage + 2 个 ToolMessage（执行结果）。
    - Router：检测无工具但还有任务 → "scout"（继续）。
  - **scout (Drone2)**：同上，手动模拟执行成功。`task_queue: []`，`completed_tasks: [Drone1, Drone2]`。
  - **Router**：任务空 → END。完美结束，无循环。
  
- **第二次运行（invoke）**：
  - **supervisor**：为什么又运行？因为 `app.invoke(initial_state)` 是**全新实例**，从初始空状态重启（`task_queue: []`），所以 LLM 又分解一次任务（重复输出）。
  - **scout (Drone1)**：这次 LLM 成功生成工具调用（Tool calls: 2）！可能是 prompt 强化生效，或模型随机性。Router → "tool_executor" → 执行 takeoff + fly_to（AirSim 打印）。
  - **scout (Drone2)**：LLM 仍无工具 → 手动模拟执行。
  - **结束**：同上，任务完成。最终摘要显示剩余 0，完成 2 个，消息 7 条（累积日志）。

**结论**：**正确！** 两次运行都实现了使命目标：
- 无人机起飞并到达（AirSim 可见）。
- 状态更新无误（队列清空，完成追加）。
- Fallback 机制（手动模拟）确保了鲁棒性，即使 LLM 偶尔“罢工”。
- 无 KeyError 或 TypeError，路由/边定义稳。

如果你在 AirSim 视图中看到两架无人机从原点 (0,0,0) 移动到目标（z=-10 表示 10m 高度），那就是物理执行成功了。恭喜，实验验证通过！

#### 2. **潜在小问题 & 为什么出现**
- **第二次 supervisor 重复**：`app.stream()` 和 `app.invoke()` 是独立执行，每次从 `initial_state` 重置（空队列）。这是预期行为，但如果想单次完整运行，注释掉 `app.invoke` 只用 stream，或用 `app.invoke` 替换 stream（它返回最终状态，而非迭代事件）。
  - 修复建议：运行时注释 `final_state = app.invoke(...)` 行，只看 stream 输出。或加 `print("--- 第二次运行（invoke） ---")` 区分。
- **LLM 工具调用不稳定**（Drone1 第一次失败，第二次成功；Drone2 始终失败）：
  - 原因：`doubao-seed-1-6-250615` 模型工具支持可能弱（中文 prompt + bind_tools 兼容性）。日志显示 "LLM 输出 for DroneX: ..."（截断），可能输出纯文本如 “我将起飞” 而非结构化调用。
  - 影响：不致命（手动 fallback 救场），但理想中全靠 LLM 更优雅。
- **消息累积**：第二次 messages 有 7 条（包括第一次的），因为 invoke 是新图。但 Annotated[add] 正确追加，无覆盖。

# 实验执行与教学意义

当您运行此脚本时，请同时打开 AirSim 窗口。您将看到 `Drone1` 和 `Drone2` 依次（或并行，取决于工具的异步实现）起飞，并飞向监督者为它们分配的目标航点。

与此同时，控制台会打印出每个节点的输入和输出，清晰地展示了系统的“思考”过程——  
1. **任务分解**  
2. **工具调用请求**  
3. **工具执行结果**

## 核心设计理念：ReAct 框架的物理实现

该实验的设计，将智能体的两个核心能力明确分离开来：

- **推理（Reason）**：由节点中的 LLM 驱动，负责决策、规划和协调。
- **行动（Act）**：由封装了 AirSim API 的工具函数执行，负责与物理世界交互。

这正是对 **ReAct（Reason + Act）** 这一重要智能体框架的**物理实现**。

> 🎯 **引用说明**：[3] ReAct: Synergizing Reasoning and Acting in Language Models

## 教学价值

通过本实验，学生不再是仅仅“让无人机飞起来”，而是在一个高保真的 3D 模拟环境中，亲手实现了一个核心的 **智能体推理-行动循环**。

这种将抽象 AI 概念具象化的体验，具有以下优势：

- ✅ **加深理论理解**：亲眼见证 LLM 如何做出决策并驱动真实动作。
- ✅ **提升动手能力**：掌握多智能体编排、状态管理与外部系统集成。
- ✅ **增强知识迁移性**：所学模式可迁移到机器人、自动驾驶、工业自动化等领域。

> 💡 总结：本实验不仅验证了技术可行性，更是一种“做中学”（Learning by Doing）的典范，有效连接了人工智能理论与现实世界应用。

#  高级主题与未来方向

本章所介绍的系统仅仅是通往真正自主无人机集群之路的开端。在将这些系统从模拟推向现实的过程中，研究人员和工程师们正面临着一系列严峻的挑战和令人兴奋的研究方向。

## 从模拟到现实的迁移 (Sim-to-Real Transfer)

在 AirSim 中完美运行的系统，在部署到物理硬件上时可能会遇到各种问题。真实世界的传感器存在噪声，无线通信存在延迟和丢包，电机和电池的物理特性也与模拟模型有差异。解决“模拟到现实”的鸿沟是机器人学领域一个长期存在的核心挑战。

## 缓解 LLM 在机器人技术中的局限性

### 接地 (Grounding)
如何确保 LLM 的符号化“理解”与无人机传感器感知的物理现实紧密相连？一个关键的研究方向是利用实时传感器数据来验证 LLM 的计划，或纠正其对世界状态的认知偏差，从而实现语义与感知的对齐。

### 安全性与幻觉
如果一个智能体“幻觉”出一个飞向建筑物的指令，后果将是灾难性的。因此，必须设计多层安全保障机制，包括：
- 在执行前对 LLM 生成的指令进行逻辑和安全性验证；
- 设置地理围栏等硬性安全边界；
- 实现可靠的“人在回路”（Human-in-the-loop）监督和干预机制，确保人类操作员能够随时接管控制权。

## 机载LLM vs. 云端LLM

这是一个关键的架构权衡：

- **云端 LLM**：可以利用最强大的大语言模型，具备更强的推理和规划能力，但会引入显著的通信延迟，这对于需要快速反应的无人机来说可能是致命的。
- **机载 LLM（边缘计算）**：将经过优化的、更小的轻量级模型部署在无人机上，可实现低延迟、高可靠性的本地决策，但受限于无人机的计算能力和能耗。

未来的发展趋势可能是**云-边协同的混合架构**：云端负责长期战略规划和知识支持，边缘端负责实时响应和安全控制，两者通过高效通信协议协同工作。

## 研究前沿

当前的研究正在探索更为复杂的协同智能系统，主要方向包括：

- **动态组织架构选择**：使用 LLM 根据实时任务参数（如任务复杂度、集群规模、通信质量）动态地为无人机集群选择最优的协作模式，例如集中式、层级式或分布式架构。
  
- **LLM 与多智能体强化学习（MARL）的融合**：构建混合系统，结合 LLM 的语义理解与任务分解能力，以及 MARL 在密集奖励信号下优化协作策略的能力，充分发挥两者的优势。

这些前沿探索正在不断弥合学术研究与工业实践之间的差距，推动自主无人机系统向更高层次的智能化、自适应性和可靠性发展。